In [32]:
# ===== Cell 1: installs (run once) =====
!pip -q install ultralytics opencv-contrib-python scikit-learn pytorchvideo

# ===== Imports =====
import os
from pathlib import Path

import numpy as np
import cv2

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from pytorchvideo.models.hub import x3d_s
from torchvision import models
from ultralytics import YOLO

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Torch: 2.5.1 | CUDA: True


In [33]:
# ===== Cell 2: config & helpers =====

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- Paths (adjust ROOT if needed) ----
ROOT = Path(r"E:\Graduation Project")  # CHANGE if different
EVAL_ROOT = ROOT / "eval_data"
MODELS_DIR = ROOT / "models"

print("EVAL_ROOT:", EVAL_ROOT)
print("MODELS_DIR:", MODELS_DIR)

# ---- Data / model config ----
NUM_CLASSES = 2
T = 13          # same as training
SIZE = 160      # same as training

# Thresholds (from your main test-set sweeps)
THRESH_X3D = 0.6
THRESH_MNET = 0.55

# Label mapping
CLASS_TO_LABEL = {"non_violence": 0, "violence": 1}

# Normalization (ImageNet)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_frame(bgr, size=SIZE):
    """
    Same as your training/eval pipeline:
      - resize so short side = 1.14*size
      - center crop to (size, size)
      - BGR -> RGB
      - normalize by ImageNet mean/std
      - return CHW float32
    """
    h, w = bgr.shape[:2]
    scale = int(size * 1.14)
    short = min(h, w)
    r = scale / float(short)
    nw, nh = int(round(w * r)), int(round(h * r))
    resized = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)

    y = max((nh - size) // 2, 0)
    x = max((nw - size) // 2, 0)
    crop = resized[y:y + size, x:x + size]
    if crop.shape[0] != size or crop.shape[1] != size:
        crop = cv2.resize(crop, (size, size), interpolation=cv2.INTER_LINEAR)

    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
    chw = np.transpose(rgb, (2, 0, 1))  # (C, H, W)
    return chw.astype(np.float32)


def sample_indices(n_frames, T):
    """
    Get T indices from a video of length n_frames.
    Center clip if n_frames > T, otherwise interpolate.
    """
    if n_frames <= 0:
        return []

    if n_frames <= T:
        return np.linspace(0, n_frames - 1, T).astype(int)

    start = (n_frames - T) // 2
    return np.arange(start, start + T)


def load_eval_samples():
    samples = []
    for cls_name, label in CLASS_TO_LABEL.items():
        folder = EVAL_ROOT / cls_name
        if not folder.exists():
            print(f"[WARN] Missing folder: {folder}")
            continue
        for vid in sorted(folder.glob("*")):
            if vid.suffix.lower() not in [".mp4", ".avi", ".mov", ".mkv"]:
                continue
            samples.append((vid, label))
    print(f"Total eval videos: {len(samples)}")
    return samples

Using device: cuda
EVAL_ROOT: E:\Graduation Project\eval_data
MODELS_DIR: E:\Graduation Project\models


In [34]:
# ===== Cell 3: define models & load weights =====

# ---- X3D-S model ----
model_x3d = x3d_s(pretrained=False)
in_features = model_x3d.blocks[5].proj.in_features
model_x3d.blocks[5].proj = nn.Linear(in_features, NUM_CLASSES)
state_x3d = torch.load(MODELS_DIR / "x3d_s_best.pth", map_location=DEVICE)
model_x3d.load_state_dict(state_x3d)
model_x3d = model_x3d.to(DEVICE)
model_x3d.eval()
print("Loaded X3D-S best weights.")


# ---- MobileNetClip wrapper ----
class MobileNetClip(nn.Module):
    """
    Wraps MobileNetV2 to handle [B, T, C, H, W]:
      - Flatten frames into batch
      - Run MobileNet on each frame
      - Average logits across T
    """
    def __init__(self, backbone, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = backbone
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        logits_frame = self.backbone(x)         # [B*T, num_classes]
        logits_frame = logits_frame.view(B, T, NUM_CLASSES)
        logits = logits_frame.mean(dim=1)       # [B, num_classes]
        return logits


backbone = models.mobilenet_v2(weights=None)
model_mnet = MobileNetClip(backbone, num_classes=NUM_CLASSES).to(DEVICE)
state_mnet = torch.load(MODELS_DIR / "mobilenet_clip_best.pth", map_location=DEVICE)
model_mnet.load_state_dict(state_mnet)
model_mnet.eval()
print("Loaded MobileNetClip best weights.")

C:\Users\abdal\AppData\Local\Temp\ipykernel_41708\3693739284.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_x3d = torch.load(MODELS_DIR / "x3d_s_best.pth", map_lo

Loaded X3D-S best weights.
Loaded MobileNetClip best weights.


C:\Users\abdal\AppData\Local\Temp\ipykernel_41708\3693739284.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_mnet = torch.load(MODELS_DIR / "mobilenet_clip_best.p

In [35]:
# ===== Cell 4: load YOLOv8n for person counting =====

people_model = YOLO("yolov8n.pt")  # downloads on first run
print("YOLOv8n loaded with classes:", people_model.names)

YOLOv8n loaded with classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 

In [36]:
# ===== Cell 5: per-video analysis =====

def analyze_video_both_models(video_path, model_x3d, model_mnet, people_model, device):
    """
    For a single video:
      - read all frames via OpenCV
      - count max number of people in any frame using YOLOv8n
      - sample T frames for the models
      - run X3D-S and MobileNetClip
    Returns:
      p_x3d (float), p_mnet (float), max_people (int)
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[WARN] Cannot open {video_path}")
        return None, None, 0

    frames_bgr = []
    max_people = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # --- people counting with YOLOv8n ---
        results = people_model(frame, verbose=False)
        frame_people = 0
        for r in results:
            boxes = getattr(r, "boxes", None)
            if boxes is None:
                continue
            for box in boxes:
                cls_id = int(box.cls[0])
                label = people_model.names.get(cls_id, str(cls_id))
                conf = float(box.conf[0])
                if label.lower() == "person" and conf >= 0.4:
                    frame_people += 1
        max_people = max(max_people, frame_people)

        frames_bgr.append(frame)

    cap.release()

    n_frames = len(frames_bgr)
    if n_frames == 0:
        print(f"[WARN] No frames read from {video_path}")
        return None, None, max_people

    idxs = sample_indices(n_frames, T)
    if len(idxs) == 0:
        return None, None, max_people

    frames_chw = [preprocess_frame(frames_bgr[i], SIZE) for i in idxs]
    # Ensure exactly T frames (in case of any weirdness)
    while len(frames_chw) < T:
        frames_chw.append(frames_chw[-1])

    clip_np = np.stack(frames_chw, axis=0)  # [T, C, H, W]

    # MobileNet: [1, T, C, H, W]
    clip_mnet = torch.from_numpy(clip_np).unsqueeze(0).to(device)  # float32

    # X3D: [1, C, T, H, W]
    clip_x3d_np = np.transpose(clip_np, (1, 0, 2, 3))  # [C, T, H, W]
    clip_x3d = torch.from_numpy(clip_x3d_np).unsqueeze(0).to(device)

    with torch.no_grad():
        logits_x3d = model_x3d(clip_x3d)
        probs_x3d = torch.softmax(logits_x3d, dim=1)[:, 1]  # P(violence)
        p_x3d = float(probs_x3d.item())

        logits_mnet = model_mnet(clip_mnet)
        probs_mnet = torch.softmax(logits_mnet, dim=1)[:, 1]
        p_mnet = float(probs_mnet.item())

    return p_x3d, p_mnet, max_people

In [37]:
# ===== Cell 6: run eval & compute metrics =====

samples = load_eval_samples()
if not samples:
    raise SystemExit("No eval videos found. Check EVAL_ROOT.")

y_true = []
y_pred_x3d = []
y_pred_mnet = []
max_people_list = []

for vid_path, label in samples:
    print(f"\nVideo: {vid_path}")
    p_x3d, p_mnet, max_people = analyze_video_both_models(
        vid_path, model_x3d, model_mnet, people_model, DEVICE
    )

    if p_x3d is None or p_mnet is None:
        print("  [skip] could not compute model probabilities")
        continue

    pred_x3d = 1 if p_x3d >= THRESH_X3D else 0
    pred_mnet = 1 if p_mnet >= THRESH_MNET else 0

    print(f"  X3D-S violence_prob : {p_x3d:.3f} -> pred {pred_x3d}")
    print(f"  MobileNet violence_prob : {p_mnet:.3f} -> pred {pred_mnet}")
    print(f"  max_people_in_video : {max_people}")

    y_true.append(label)
    y_pred_x3d.append(pred_x3d)
    y_pred_mnet.append(pred_mnet)
    max_people_list.append(max_people)

if not y_true:
    raise SystemExit("No valid predictions produced.")

y_true = np.array(y_true)
y_pred_x3d = np.array(y_pred_x3d)
y_pred_mnet = np.array(y_pred_mnet)
max_people_list = np.array(max_people_list)

# ---- Metrics for X3D-S ----
print("\n================ X3D-S metrics ================")
acc_x = accuracy_score(y_true, y_pred_x3d)
prec_x = precision_score(y_true, y_pred_x3d, zero_division=0)
rec_x = recall_score(y_true, y_pred_x3d, zero_division=0)
f1_x = f1_score(y_true, y_pred_x3d, zero_division=0)

print(f"Accuracy : {acc_x:.4f}")
print(f"Precision: {prec_x:.4f}")
print(f"Recall   : {rec_x:.4f}")
print(f"F1       : {f1_x:.4f}")
print("Confusion matrix (X3D-S):")
print(confusion_matrix(y_true, y_pred_x3d))
print(
    classification_report(
        y_true,
        y_pred_x3d,
        target_names=["NonViolence", "Violence"],
        zero_division=0,
        digits=4,
    )
)

# ---- Metrics for MobileNetClip ----
print("\n================ MobileNetClip metrics ================")
acc_m = accuracy_score(y_true, y_pred_mnet)
prec_m = precision_score(y_true, y_pred_mnet, zero_division=0)
rec_m = recall_score(y_true, y_pred_mnet, zero_division=0)
f1_m = f1_score(y_true, y_pred_mnet, zero_division=0)

print(f"Accuracy : {acc_m:.4f}")
print(f"Precision: {prec_m:.4f}")
print(f"Recall   : {rec_m:.4f}")
print(f"F1       : {f1_m:.4f}")
print("Confusion matrix (MobileNetClip):")
print(confusion_matrix(y_true, y_pred_mnet))
print(
    classification_report(
        y_true,
        y_pred_mnet,
        target_names=["NonViolence", "Violence"],
        zero_division=0,
        digits=4,
    )
)

# Optional: some quick stats on people counts
print("\nPeople count stats on eval set:")
print("  mean max_people:", float(max_people_list.mean()))
print("  max max_people :", int(max_people_list.max()))

Total eval videos: 120

Video: E:\Graduation Project\eval_data\non_violence\1.mp4
  X3D-S violence_prob : 0.580 -> pred 0
  MobileNet violence_prob : 0.997 -> pred 1
  max_people_in_video : 4

Video: E:\Graduation Project\eval_data\non_violence\10.mp4
  X3D-S violence_prob : 0.269 -> pred 0
  MobileNet violence_prob : 0.159 -> pred 0
  max_people_in_video : 5

Video: E:\Graduation Project\eval_data\non_violence\11.mp4
  X3D-S violence_prob : 0.313 -> pred 0
  MobileNet violence_prob : 0.226 -> pred 0
  max_people_in_video : 5

Video: E:\Graduation Project\eval_data\non_violence\12.mp4
  X3D-S violence_prob : 0.276 -> pred 0
  MobileNet violence_prob : 0.409 -> pred 0
  max_people_in_video : 5

Video: E:\Graduation Project\eval_data\non_violence\13.mp4
  X3D-S violence_prob : 0.688 -> pred 1
  MobileNet violence_prob : 0.981 -> pred 1
  max_people_in_video : 5

Video: E:\Graduation Project\eval_data\non_violence\14.mp4
  X3D-S violence_prob : 0.273 -> pred 0
  MobileNet violence_prob : 